# Partie : Nettoyage des donnees :

In [14]:
#importation
import pandas as pd
import numpy as np

## Chargemment des donnees :

In [3]:
df = pd.read_csv("dataset1.csv")
print(df.head())

             nom         poste  age nationalite                 club  \
0   Lamine Yamal  Right Winger   17       Spain         FC Barcelona   
1    Bukayo Saka  Right Winger   23     England           Arsenal FC   
2  Michael Olise  Right Winger   23      France        Bayern Munich   
3        Rodrygo  Right Winger   23      Brazil          Real Madrid   
4    Désiré Doué  Right Winger   19      France  Paris Saint-Germain   

  valeur_marchande  matches_played  goals  own_goals  assists  yellow_cards  \
0         €200.00m              54     21          0       24             5   
1         €150.00m              38     10          0        5             3   
2         €130.00m              57     21          0       28             7   
3          €90.00m              54     10          0       11             4   
4          €90.00m              54     19          0       16             3   

   red_cards  
0          0  
1          0  
2          0  
3          0  
4          0  


## 1. Supprimer doublons et valeurs manquantes :

In [4]:
df = df.dropna(subset=["nom"])
avant = df.shape[0]
df = df.drop_duplicates(subset=["nom", "club"], keep="first")
print(f"Doublons supprimés : {avant - df.shape[0]}")
print(f"Après nettoyage : {df.shape}")

Doublons supprimés : 23
Après nettoyage : (2136, 12)


## 2. Convertir valeur_marchande (€50.00m -> 50000000) :

In [9]:
def convertir_valeur(valeur):
    if pd.isna(valeur):
        return np.nan
    valeur = str(valeur).replace("€", "").strip()
    if "m" in valeur.lower():
        return float(valeur.lower().replace("m", "")) * 1_000_000
    elif "k" in valeur.lower():
        return float(valeur.lower().replace("k", "")) * 1_000
    try:
        return float(valeur)
    except ValueError:
        return np.nan

df["valeur_marchande_eur"] = df["valeur_marchande"].apply(convertir_valeur)

# 3b. Transformation logarithmique de la valeur marchande
# ---------------------------------------------------------
# La valeur marchande est très asymétrique (quelques stars à 200M€, la majorité
# à 1-20M€). Le log réduit l'écrasement que ces valeurs extrêmes provoqueraient
# dans un modèle de régression, sans supprimer aucune donnée.
df["valeur_marchande_log"] = np.log1p(df["valeur_marchande_eur"])
print(f"\nExemple de transformation log :")
print(df[["nom", "valeur_marchande_eur", "valeur_marchande_log"]]
      .sort_values("valeur_marchande_eur", ascending=False).head(3).to_string())




Exemple de transformation log :
                 nom  valeur_marchande_eur  valeur_marchande_log
0       Lamine Yamal           200000000.0             19.113828
1488  Erling Haaland           200000000.0             19.113828
1489   Kylian Mbappé           200000000.0             19.113828


## 3. Créer des indicateurs pour les grandes nations du football:

In [7]:
# Plutôt qu'un regroupement large par continent, on isole spécifiquement les
# grandes nations du football historiquement associées à une forte valorisation
# de leurs joueurs. Chaque colonne vaut 1 si le joueur a cette nationalité, 0 sinon
# (un joueur d'une autre nationalité aura donc un 0 dans toutes ces colonnes).

grandes_nations = ["Brazil", "England", "Spain", "France", "Argentina", "Germany"]

for pays in grandes_nations:
    df[f"nat_{pays}"] = (df["nationalite"] == pays).astype(int)

print(f"\nColonnes créées : {[f'nat_{p}' for p in grandes_nations]}")
for pays in grandes_nations:
    print(f"  nat_{pays} : {df[f'nat_{pays}'].sum()} joueurs")



Colonnes créées : ['nat_Brazil', 'nat_England', 'nat_Spain', 'nat_France', 'nat_Argentina', 'nat_Germany']
  nat_Brazil : 168 joueurs
  nat_England : 151 joueurs
  nat_Spain : 139 joueurs
  nat_France : 131 joueurs
  nat_Argentina : 115 joueurs
  nat_Germany : 96 joueurs


## 4. Encodage des variables catégorielles :

In [8]:
df_poste_encoded = pd.get_dummies(df["poste"], prefix="poste")
df = pd.concat([df, df_poste_encoded], axis=1)
print(f"\nColonnes créées pour le poste : {list(df_poste_encoded.columns)}")


Colonnes créées pour le poste : ['poste_Attacking Midfield', 'poste_Central Midfield', 'poste_Centre-Back', 'poste_Centre-Forward', 'poste_Defensive Midfield', 'poste_Goalkeeper', 'poste_Left Midfield', 'poste_Left Winger', 'poste_Left-Back', 'poste_Right Midfield', 'poste_Right Winger', 'poste_Right-Back']


## 5. Détection des outliers [Valeurs aberrante] (méthode IQR):

In [12]:
def detecter_outliers_iqr(serie):
    q1 = serie.quantile(0.25)
    q3 = serie.quantile(0.75)
    iqr = q3 - q1
    borne_basse = q1 - 1.5 * iqr
    borne_haute = q3 + 1.5 * iqr
    return (serie < borne_basse) | (serie > borne_haute)

colonnes_a_verifier = ["valeur_marchande_eur", "matches_played", "goals", "assists", "age"]

print("\n=== Détection des outliers (IQR) ===")
for col in colonnes_a_verifier:
    mask_outliers = detecter_outliers_iqr(df[col])
    print(f"{col} : {mask_outliers.sum()} outliers détectés")

df["outlier_valeur"] = detecter_outliers_iqr(df["valeur_marchande_eur"])



=== Détection des outliers (IQR) ===
valeur_marchande_eur : 99 outliers détectés
matches_played : 23 outliers détectés
goals : 127 outliers détectés
assists : 71 outliers détectés
age : 6 outliers détectés


In [13]:
df.to_csv("players_clean.csv", index=False, sep=",", encoding="utf-8-sig")
print(f"\n=== Fichier final : players_clean.csv — {df.shape[0]} lignes, {df.shape[1]} colonnes ===")


=== Fichier final : players_clean.csv — 2136 lignes, 38 colonnes ===
